In [ ]:
!git clone https://github.com/JinYSun/D-GCAN.git
%cd D-GCAN

Cloning into 'D-GCAN'...
remote: Enumerating objects: 493, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 493 (delta 31), reused 30 (delta 14), pack-reused 441 (from 1)
Receiving objects: 100% (493/493), 7.53 MiB | 14.55 MiB/s, done.
Resolving deltas: 100% (278/278), done.
/content/D-GCAN


In [ ]:
!pip install rdkit -q
# PyTorch is already preinstalled in Colab — no need to reinstall
import torch
print(torch.__version__, torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 63.2 MB/s eta 0:00:00
2.11.0+cpu False


In [ ]:
!ls dataset/
!ls DGCAN/
!head -5 dataset/data_train.txt

bRo5.txt  data_test.txt  data_train.txt  nonUS.txt  withdrawn.txt
DGCAN.py  __init__.py  preprocess.py  run.py	Tutorial.ipynb
dict	  predict.py   results	      train.py
CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@H](CCC(O)=O)NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)[C@H](CC(O)=O)NC(=O)CNC(=O)[C@H](CC(N)=O)NC(=O)CNC(=O)CNC(=O)CNC(=O)CNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCNC(N)=N)NC(=O)[C@@H]1CCCN1C(=O)[C@H](N)CC1=CC=CC=C1)C(=O)N1CCC[C@H]1C(=O)N[C@@H](CCC(O)=O)C(=O)N[C@@H](CCC(O)=O)C(=O)N[C@@H](CC1=CC=C(O)C=C1)C(=O)N[C@@H](CC(C)C)C(O)=O  1
CCCCCCCN(CC)CCCC(O)C1=CC=C(NS(C)(=O)=O)C=C1  1
CC[C@H]1[C@@H](CC2=CN=CN2C)COC1=O  1
CN(C)S(=O)(=O)C1=CC2=C(SC3=CC=CC=C3\C2=C\CCN2CCN(C)CC2)C=C1  1
CC(=O)NO  1


In [ ]:
import os
os.makedirs('model', exist_ok=True)
os.makedirs('output', exist_ok=True)
print(os.listdir('.'))

['solgan.png', 'DGCAN', 'LICENSE', 'Test', 'Discussion', 'Tutorial.ipynb', 'screening', 'output', 'model', 'dataset', '.gitignore', 'README.md', '.git']


In [ ]:
%cd DGCAN
import train

test = train.train(
    '../dataset/bRo5.txt',
    radius=1,
    dim=52,
    layer_hidden=4,
    layer_output=10,
    dropout=0.45,
    batch_train=8,
    batch_test=8,
    lr=3e-4,
    lr_decay=0.85,
    decay_interval=25,
    iteration=10,      # ← start small, just to test the pipeline works
    N=5000,
    dataset_train='../dataset/data_train.txt'
)

[Errno 2] No such file or directory: 'DGCAN'
/content/D-GCAN/DGCAN
The code uses a GPU!
----------------------------------------------------------------------------------------------------
Just a moment......
----------------------------------------------------------------------------------------------------
../dataset/data_train.txt
../dataset/bRo5.txt
The preprocess has finished!
----------------------------------------------------------------------------------------------------
Creating a model.
# of model parameters: 311698
----------------------------------------------------------------------------------------------------
Start training.
The result is saved in the output directory every epoch!
The training will finish in about 0 hours 1 minutes.
----------------------------------------------------------------------------------------------------
Epoch	Time(sec)	Loss_train	Loss_test	AUC_train	AUC_test
1	8.100747198000136	318.32105273008347	20.28933635354042	0.6240720548187837	0.5173

In [ ]:
test = train.train(
    '../dataset/bRo5.txt',
    radius=1,
    dim=52,
    layer_hidden=4,
    layer_output=10,
    dropout=0.45,
    batch_train=8,
    batch_test=8,
    lr=3e-4,
    lr_decay=0.85,
    decay_interval=25,
    iteration=140,     # ← full run now
    N=5000,
    dataset_train='../dataset/data_train.txt'
)

The code uses a GPU!
----------------------------------------------------------------------------------------------------
Just a moment......
----------------------------------------------------------------------------------------------------
../dataset/data_train.txt
../dataset/bRo5.txt
The preprocess has finished!
----------------------------------------------------------------------------------------------------
Creating a model.
# of model parameters: 311698
----------------------------------------------------------------------------------------------------
Start training.
The result is saved in the output directory every epoch!
The training will finish in about 0 hours 19 minutes.
----------------------------------------------------------------------------------------------------
Epoch	Time(sec)	Loss_train	Loss_test	AUC_train	AUC_test
1	8.142978486999937	318.32105273008347	20.28933635354042	0.6240720548187837	0.5173745173745173
2	15.921784003000084	279.60529312491417	16.0917206108

In [ ]:
!cat DGCAN.py

# -*- coding: utf-8 -*-
"""
Created on Wed Apr 27 18:04:58 2022

@author:Jinyu-Sun
"""

#coding=utf-8 
import timeit
import sys
import numpy as np
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pickle
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import confusion_matrix
import preprocess as pp
import pandas as pd
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = torch.device('cuda')
   
else:
    device = torch.device('cpu')
    
torch.cuda.empty_cache()
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout, alpha, concat=True):
        super(GraphAttentionLayer, self).__init__()
        self.dropout = dropout 
        self.concat = concat
        self.in_features = in_features   #dim of input feature
        self.out_features = out_features #dim of output feature
        self.alpha = alpha               # negative_slope leak

In [ ]:
!ls model/

ls: cannot access 'model/': No such file or directory


In [ ]:
!pwd
!ls

/content/D-GCAN/DGCAN
DGCAN.py  __init__.py  preprocess.py  results  train.py
dict	  predict.py   __pycache__    run.py   Tutorial.ipynb


In [ ]:
import os
os.makedirs('model', exist_ok=True)
os.makedirs('output', exist_ok=True)

import train
test = train.train(
    '../dataset/bRo5.txt',
    radius=1, dim=52, layer_hidden=4, layer_output=10,
    dropout=0.45, batch_train=8, batch_test=8,
    lr=3e-4, lr_decay=0.85, decay_interval=25,
    iteration=140, N=5000,
    dataset_train='../dataset/data_train.txt'
)

The code uses a CPU...
----------------------------------------------------------------------------------------------------
Just a moment......
----------------------------------------------------------------------------------------------------
../dataset/data_train.txt
CCCCCCCN(CC)CCCC(O)C1=CC=C(NS(C)(=O)=O)C=C1
CC[C@H]1[C@@H](CC2=CN=CN2C)COC1=O
CN(C)S(=O)(=O)C1=CC2=C(SC3=CC=CC=C3\C2=C\CCN2CCN(C)CC2)C=C1
CC(=O)NO
CN1C2=C(C=C(Cl)C=C2)C(=NCC1=O)C1=CC=CC=C1
CC1=CC(=NO1)C(=O)NNCC1=CC=CC=C1
[H][C@]12SCC(CSC3=NN=NN3C)=C(N1C(=O)[C@H]2NC(=O)[C@H](OC=O)C1=CC=CC=C1)C(O)=O
CN1C=NC2=C1C(=O)N(CCCCC(C)=O)C(=O)N2C
COC(=O)N[C@@H](C(C)C)C(=O)N1CCC[C@H]1C(=O)NC1=CC=C(C=C1)[C@@H]1CC[C@H](N1C1=CC=C(C=C1)C(C)(C)C)C1=CC=C(NC(=O)[C@@H]2CCCN2C(=O)[C@@H](NC(=O)OC)C(C)C)C=C1
OC[C@H]1O[C@@](CO)(OC[C@@]2(OC[C@@]3(OC[C@@]4(OC[C@@]5(OC[C@@]6(OC[C@@]7(OC[C@@]8(OC[C@@]9(OC[C@@]%10(OC[C@@]%11(OC[C@@]%12(OC[C@@]%13(OC[C@@]%14(OC[C@@]%15(OC[C@@]%16(OC[C@@]%17(OC[C@@]%18(OC[C@@]%19(OC[C@@]%20(OC[C@@]%21(OC[C@@]%22(OC[C@

ValueError: need at least one array to concatenate

In [ ]:
!ls dict/
!ls ../dataset/
!wc -l ../dataset/data_train.txt
!wc -l ../dataset/bRo5.txt

-atom_dict.pickle  -edge_dict.pickle
-bond_dict.pickle  -fingerprint_dict.pickle
bRo5.txt  data_test.txt  data_train.txt  nonUS.txt  withdrawn.txt
3843 ../dataset/data_train.txt
259 ../dataset/bRo5.txt


In [ ]:
!grep -n "dict" preprocess.py | head -30

9:from collections import defaultdict
16:atom_dict = defaultdict(lambda: len(atom_dict))
17:bond_dict = defaultdict(lambda: len(bond_dict))
18:fingerprint_dict = defaultdict(lambda: len(fingerprint_dict))
19:edge_dict = defaultdict(lambda: len(edge_dict))
21:def create_atoms(mol, atom_dict):
30:    atoms = [atom_dict[a] for a in atoms]
34:def create_ijbonddict(mol, bond_dict):
35:    """Create a dictionary, in which each key is a node ID
40:    i_jbond_dict = defaultdict(lambda: [])
43:        bond = bond_dict[str(b.GetBondType())]
44:        i_jbond_dict[i].append((j, bond))
45:        i_jbond_dict[j].append((i, bond))
46:    return i_jbond_dict
49:def extract_fingerprints(radius, atoms, i_jbond_dict,
50:                         fingerprint_dict, edge_dict):
56:        nodes = [fingerprint_dict[a] for a in atoms]
60:        i_jedge_dict = i_jbond_dict
68:            for i, j_edge in i_jedge_dict.items():
71:                nodes_.append(fingerprint_dict[fingerprint])
76:            i_

In [ ]:
!grep -n "dump\|pickle\|filename\|\.pkl\|dict_dir\|path" preprocess.py

95:def create_testdataset(filename,path,dataname,property):
96:    dir_dataset = path+dataname
97:    print(filename)
100:                with open(dir_dataset + filename, 'r') as f:
127:        with open(dir_dataset + filename, 'r') as f:
157:def create_dataset(filename,path,dataname):
158:    dir_dataset = path+dataname
159:    print(filename)
162:        with open(dir_dataset + filename, 'r') as f:
166:        with open(dir_dataset + filename, 'r') as f:


In [ ]:
import pickle
with open('dict/-fingerprint_dict.pickle', 'rb') as f:
    fp_dict = pickle.load(f)
print(len(fp_dict))
print(list(fp_dict.items())[:5])

520
[((np.int32(0), ((np.int32(0), 0), (np.int32(5), 0), (np.int32(5), 0), (np.int32(5), 0))), 0), ((np.int32(0), ((np.int32(0), 0), (np.int32(0), 0), (np.int32(5), 0), (np.int32(5), 0))), 1), ((np.int32(0), ((np.int32(0), 0), (np.int32(1), 0), (np.int32(5), 0), (np.int32(5), 0))), 2), ((np.int32(1), ((np.int32(0), 0), (np.int32(0), 0), (np.int32(0), 0))), 3), ((np.int32(0), ((np.int32(0), 0), (np.int32(2), 0), (np.int32(3), 0), (np.int32(5), 0))), 4)]


In [ ]:
dir_dataset = path + dataname

NameError: name 'path' is not defined

In [ ]:
!cat train.py

# -*- coding: utf-8 -*-
"""
Created on Wed Apr 27 20:09:31 2022

@author:Jinyu-Sun
"""

import timeit
import sys
import numpy as np
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pickle
from sklearn.metrics import roc_auc_score, roc_curve,auc
from sklearn.metrics import confusion_matrix
import preprocess as pp
import pandas as pd
import matplotlib.pyplot as plt
from DGCAN import MolecularGraphNeuralNetwork,Trainer,Tester

def metrics(cnf_matrix):
    '''Evaluation Metrics'''
    tn = cnf_matrix[0, 0]
    tp = cnf_matrix[1, 1]
    fn = cnf_matrix[1, 0]
    fp = cnf_matrix[0, 1]

    bacc = ((tp / (tp + fn)) + (tn / (tn + fp))) / 2  # balance accurance
    pre = tp / (tp + fp)  # precision/q+
    rec = tp / (tp + fn)  # recall/se
    sp = tn / (tn + fp)
    q_ = tn / (tn + fn)
    f1 = 2 * pre * rec / (pre + rec)  # f1score
    mcc = ((tp * tn) - (fp * fn)) / math.sqrt(
        (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn

In [ ]:
import preprocess as pp
dataset_train_test = pp.create_dataset('../dataset/data_train.txt', '', '')
print(len(dataset_train_test))
print(dataset_train_test[:2] if len(dataset_train_test) > 0 else "EMPTY!")

../dataset/data_train.txt
CCCCCCCN(CC)CCCC(O)C1=CC=C(NS(C)(=O)=O)C=C1
CC[C@H]1[C@@H](CC2=CN=CN2C)COC1=O
CN(C)S(=O)(=O)C1=CC2=C(SC3=CC=CC=C3\C2=C\CCN2CCN(C)CC2)C=C1
CC(=O)NO
CN1C2=C(C=C(Cl)C=C2)C(=NCC1=O)C1=CC=CC=C1
CC1=CC(=NO1)C(=O)NNCC1=CC=CC=C1
[H][C@]12SCC(CSC3=NN=NN3C)=C(N1C(=O)[C@H]2NC(=O)[C@H](OC=O)C1=CC=CC=C1)C(O)=O
CN1C=NC2=C1C(=O)N(CCCCC(C)=O)C(=O)N2C
COC(=O)N[C@@H](C(C)C)C(=O)N1CCC[C@H]1C(=O)NC1=CC=C(C=C1)[C@@H]1CC[C@H](N1C1=CC=C(C=C1)C(C)(C)C)C1=CC=C(NC(=O)[C@@H]2CCCN2C(=O)[C@@H](NC(=O)OC)C(C)C)C=C1
OC[C@H]1O[C@@](CO)(OC[C@@]2(OC[C@@]3(OC[C@@]4(OC[C@@]5(OC[C@@]6(OC[C@@]7(OC[C@@]8(OC[C@@]9(OC[C@@]%10(OC[C@@]%11(OC[C@@]%12(OC[C@@]%13(OC[C@@]%14(OC[C@@]%15(OC[C@@]%16(OC[C@@]%17(OC[C@@]%18(OC[C@@]%19(OC[C@@]%20(OC[C@@]%21(OC[C@@]%22(OC[C@@]%23(OC[C@@]%24(OC[C@@]%25(OC[C@@]%26(OC[C@@]%27(OC[C@@]%28(OC[C@@]%29(OC[C@@]%30(OC[C@@]%31(OC[C@@]%32(OC[C@@]%33(OC[C@@]%34(OC[C@@]%35(OC[C@@]%36(OC[C@@]%37(O[C@H]%38O[C@H](CO)[C@@H](O)[C@H](O)[C@H]%38O)O[C@H](CO)[C@@H](O)[C@@H]%37O)O[C@H](CO

In [ ]:
!wc -l ../dataset/data_train.txt
!grep -c $'\t' ../dataset/data_train.txt
!tail -50 ../dataset/data_train.txt | head -20

3843 ../dataset/data_train.txt
0
Cc1ccc(-c2ccc(=O)n(CCC(=O)NCCN(CC(C)C)CC(C)C)n2)cc1  0
O=C(Nc1cc(Cl)ccc1C(=O)NCc1ccco1)c1ccc(F)cc1  0
O=C1N[C@H](c2cccc(Br)c2)c2c1oc1ccc(Cl)cc1c2=O  0
COC(=O)c1cc(N)ccc1Br  0
CN(C)c1ccc(C(=O)NCc2cnn(C)c2)cc1F  0
Clc1cccc(/C=C/c2nn3c(CN4CCCCC4)nnc3s2)c1  0
CCC(=O)Nc1ccc(OCCCc2ccccc2)cc1  0
O=C1[C@@H]2CC3=c4ccccc4=N[C@@H]3[C@H](c3cccc([N+](=O)[O-])c3)N2C(=O)CN1Cc1ccncc1  0
O=C1N[C@]2(CCc3ccccc31)C[N@@H+](Cc1ccccc1)C[C@H]2C(=O)NC[C@H]1CCCCN1  0
Cc1cccc(C)c1NC(=O)c1ccc(S(=O)(=O)c2ccccc2)o1  0
COc1ccccc1C(=O)N1[C@H]2CCCC[C@@H]2C[C@@H]1C(=O)Nc1ccccc1C  0
Cc1noc(C)c1Cc1ccc(C(=O)NCc2ccccc2Cl)o1  0
Cc1cc([C@H](C)c2ccccc2)c(O)c(CN2CCN(Cc3c(C)c(C)cc([C@H](C)c4ccccc4)c3O)CC2)c1C  0
CCCn1cc(NCc2cc(OC)c(OC)cc2[N+](=O)[O-])cn1  0
CC[C@@H]1C(=O)C2(CCCCC2)C(=O)O[C@H]1c1ccccc1  0
CCCCN(C(=O)C1CCCCC1)[C@H]1CCS(=O)(=O)C1  0
Cc1cc(-c2nn(C)c(Cl)c2C[N@H+]2CC[C@H]([C@@H](C)N(C)C)CC2)no1  0
O=C(COc1ccc([N+](=O)[O-])cc1)N[C@@H]1C[C@H]2C[C@@H]1[C@H]1CCC[C@@H]12  0
Cc1c(CNCc2cccc(

In [ ]:
!grep -n "split\|readlines\|strip" preprocess.py | head -30


grep: preprocess.py: No such file or directory
